In [ ]:
# CELL 1
import pandas as pd
import io, os, ast, base64
from datasets import Dataset, load_from_disk
from PIL import Image
from tqdm import tqdm
import math, traceback

OUT_DIR = "/content"
CHECKPOINT_EVERY = 50   # saves intermediate files every N images
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
!unzip /content/autocot_final_dataset.zip

Archive:  /content/autocot_final_dataset.zip
   creating: content/
   creating: content/.config/
 extracting: content/.config/config_sentinel  
 extracting: content/.config/gce     
  inflating: content/.config/.last_update_check.json  
 extracting: content/.config/.last_opt_in_prompt.yaml  
  inflating: content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db  
   creating: content/.config/logs/
   creating: content/.config/logs/2025.11.20/
  inflating: content/.config/logs/2025.11.20/14.30.45.231815.log  
  inflating: content/.config/logs/2025.11.20/14.30.35.382199.log  
  inflating: content/.config/logs/2025.11.20/14.30.45.937471.log  
  inflating: content/.config/logs/2025.11.20/14.30.04.285207.log  
  inflating: content/.config/logs/2025.11.20/14.30.36.623222.log  
  inflating: content/.config/logs/2025.11.20/14.30.27.010422.log  
 extracting: content/.config/.last_survey_prompt.yaml  
 extracting: content/.config/active_config  
  inflating: content/.config/

In [ ]:
from datasets import load_from_disk
import pandas as pd

stage1 = load_from_disk("/content/content/content/image_stage1")
stage2 = load_from_disk("/content/content/notebook3_outputs/autocot_final_dataset")

In [ ]:
df1 = pd.DataFrame(stage1)
df1.head(5)

,question_id,image,question,answer,id,license,file_name,coco_url,height,width,date_captured,implicit_prompt,explicit_prompt,cot_prompt,ground_truth
0,000000561009.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,"[A bird perched on top of a tree branch., A br...",698182,2,000000561009.jpg,http://images.cocodataset.org/val2017/00000056...,427,640,2013-11-17 03:42:29,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,A bird perched on top of a tree branch.
1,000000343496.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,[A person walking by stop sign at street inter...,349381,1,000000343496.jpg,http://images.cocodataset.org/val2017/00000034...,393,640,2013-11-16 19:53:03,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,A person walking by stop sign at street inters...
2,000000097994.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,[Three laptop computers and a desktop computer...,140824,2,000000097994.jpg,http://images.cocodataset.org/val2017/00000009...,427,640,2013-11-21 01:43:00,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,Three laptop computers and a desktop computer ...
3,000000018150.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,[a man holding a piece of pizza in front of a ...,630212,3,000000018150.jpg,http://images.cocodataset.org/val2017/00000001...,480,640,2013-11-21 03:12:35,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,a man holding a piece of pizza in front of a kid
4,000000456292.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,"[A cat climbing into a cat door on a wall., A ...",641535,4,000000456292.jpg,http://images.cocodataset.org/val2017/00000045...,640,480,2013-11-15 21:01:23,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,A cat climbing into a cat door on a wall.


In [ ]:
df1["index"] = df1.index

In [ ]:
df2 = pd.DataFrame(stage2)
df2.head(3)

,index,file_name,ground_truth,image_load,auto_cot_caption,auto_cot_image,thumb_bytes
0,0,000000561009.jpg,A bird perched on top of a tree branch.,OK,Here's the chain-of-thought reasoning for the ...,The image shows a bird perched on a branch. Th...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
1,1,000000343496.jpg,A person walking by stop sign at street inters...,OK,"Okay, let's break down the reasoning for the c...",The image shows a street scene.\nA stop sign a...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
2,2,000000097994.jpg,Three laptop computers and a desktop computer ...,OK,Here's a breakdown of the reasoning:\n\n1. **I...,The scene is an indoor workspace with a desk. ...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...


In [ ]:
merged = df1.merge(df2, on="index", how="inner", suffixes=("_s1","_s2"))
merged.head(3)

,question_id,image,question,answer,id,license,file_name_s1,coco_url,height,width,...,explicit_prompt,cot_prompt,ground_truth_s1,index,file_name_s2,ground_truth_s2,image_load,auto_cot_caption,auto_cot_image,thumb_bytes
0,000000561009.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,"[A bird perched on top of a tree branch., A br...",698182,2,000000561009.jpg,http://images.cocodataset.org/val2017/00000056...,427,640,...,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,A bird perched on top of a tree branch.,0,000000561009.jpg,A bird perched on top of a tree branch.,OK,Here's the chain-of-thought reasoning for the ...,The image shows a bird perched on a branch. Th...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
1,000000343496.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,[A person walking by stop sign at street inter...,349381,1,000000343496.jpg,http://images.cocodataset.org/val2017/00000034...,393,640,...,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,A person walking by stop sign at street inters...,1,000000343496.jpg,A person walking by stop sign at street inters...,OK,"Okay, let's break down the reasoning for the c...",The image shows a street scene.\nA stop sign a...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
2,000000097994.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Please carefully observe the image and come up...,[Three laptop computers and a desktop computer...,140824,2,000000097994.jpg,http://images.cocodataset.org/val2017/00000009...,427,640,...,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,Three laptop computers and a desktop computer ...,2,000000097994.jpg,Three laptop computers and a desktop computer ...,OK,Here's a breakdown of the reasoning:\n\n1. **I...,The scene is an indoor workspace with a desk. ...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...


In [ ]:
merged.columns

Index(['question_id', 'image', 'question', 'answer', 'id', 'license',
       'file_name_s1', 'coco_url', 'height', 'width', 'date_captured',
       'implicit_prompt', 'explicit_prompt', 'cot_prompt', 'ground_truth_s1',
       'index', 'file_name_s2', 'ground_truth_s2', 'image_load',
       'auto_cot_caption', 'auto_cot_image', 'thumb_bytes'],
      dtype='object')

In [ ]:
clean = merged[[
    "index",
    "file_name_s1",
    "image",
    "implicit_prompt",
    "explicit_prompt",
    "cot_prompt",
    "auto_cot_caption",
    "auto_cot_image",
    "ground_truth_s1",
    "thumb_bytes"
]].copy()

# Rename columns
clean = clean.rename(columns={
    "file_name_s1": "file_name",
    "ground_truth_s1": "ground_truth"
})

clean.head()

,index,file_name,image,implicit_prompt,explicit_prompt,cot_prompt,auto_cot_caption,auto_cot_image,ground_truth,thumb_bytes
0,0,000000561009.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,Here's the chain-of-thought reasoning for the ...,The image shows a bird perched on a branch. Th...,A bird perched on top of a tree branch.,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
1,1,000000343496.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,"Okay, let's break down the reasoning for the c...",The image shows a street scene.\nA stop sign a...,A person walking by stop sign at street inters...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
2,2,000000097994.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,Here's a breakdown of the reasoning:\n\n1. **I...,The scene is an indoor workspace with a desk. ...,Three laptop computers and a desktop computer ...,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
3,3,000000018150.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,"Okay, let's break down the reasoning behind th...",A man is holding a slice of pizza towards a ch...,a man holding a piece of pizza in front of a kid,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...
4,4,000000456292.jpg,<PIL.JpegImagePlugin.JpegImageFile image mode=...,Describe the image.,"Describe all objects, colors, actions, and spa...",Think step-by-step about the image:\n1. Identi...,"Okay, let's break down the reasoning for the c...",The image shows a wall with graffiti.\nThere i...,A cat climbing into a cat door on a wall.,b'b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\...


In [ ]:
# CELL 3
def decode_thumb_bytes(raw):
    """Return bytes or None. Handles bytes, memoryview, str of form b'\\xff..' or base64."""
    if raw is None:
        return None
    # already bytes-like
    if isinstance(raw, (bytes, bytearray)):
        return bytes(raw)
    if isinstance(raw, memoryview):
        return raw.tobytes()
    if isinstance(raw, str):
        s = raw.strip()
        # if string looks like Python bytes literal, literal_eval will convert
        if s.startswith("b'") or s.startswith('b"'):
            try:
                return ast.literal_eval(s)
            except Exception:
                # try cleaning escapes
                try:
                    return s.encode("utf-8").decode("unicode_escape").encode("latin1")
                except Exception:
                    return None
        # try base64 decode
        try:
            return base64.b64decode(s)
        except Exception:
            return None
    return None

def pil_from_thumb_or_image(row):
    # prefer thumb_bytes then image object
    b = None
    if "thumb_bytes" in row and row["thumb_bytes"] is not None:
        b = decode_thumb_bytes(row["thumb_bytes"])
    if b:
        try:
            return Image.open(io.BytesIO(b)).convert("RGB")
        except Exception:
            b = None
    # fallback: use image object from stage1
    imgobj = row.get("image", None)
    if imgobj is None:
        return None
    if isinstance(imgobj, Image.Image):
        return imgobj.convert("RGB")
    if isinstance(imgobj, (bytes, bytearray)):
        try:
            return Image.open(io.BytesIO(imgobj)).convert("RGB")
        except Exception:
            return None
    return None

In [ ]:
# CELL 4
rows = []
n_images = len(clean)
print("Images to process:", n_images)

for i, r in tqdm(clean.iterrows(), total=n_images):
    row = r.to_dict()
    idx = row.get("index", i)
    fname = row.get("file_name")
    gt = row.get("ground_truth")
    # get thumbnail PIL if possible
    pil = pil_from_thumb_or_image(row)
    thumb_b64 = None
    if pil is not None:
        buf = io.BytesIO()
        pil.thumbnail((240,240))
        pil.save(buf, format="JPEG")
        thumb_b64 = base64.b64encode(buf.getvalue()).decode()
    # prompt sources
    implicit = row.get("implicit_prompt") or "Describe the image."
    explicit = row.get("explicit_prompt") or implicit
    cot_temp = row.get("cot_prompt") or "Think step-by-step about the image."
    auto_cap = row.get("auto_cot_caption") or ""
    auto_img = row.get("auto_cot_image") or ""

    variants = [
        ("implicit", implicit),
        ("explicit", explicit),
        ("cot_template", cot_temp),
        ("autocot_caption", auto_cap),
        ("autocot_image", auto_img)
    ]

    for ptype, prompt_text in variants:
        rows.append({
            "index": idx,
            "file_name": fname,
            "prompt_type": ptype,
            "generated_prompt": prompt_text,
            "ground_truth": gt,
            "thumbnail_b64": thumb_b64
        })

    # checkpoint every CHECKPOINT_EVERY images
    if (i+1) % CHECKPOINT_EVERY == 0:
        df_partial = pd.DataFrame(rows)
        ckpt_xlsx = os.path.join(OUT_DIR, f"batch_prompts_partial_{i+1}_images.xlsx")
        ckpt_ds = os.path.join(OUT_DIR, f"batch_prompts_partial_{i+1}_images_dataset")
        df_partial.to_excel(ckpt_xlsx, index=False)
        Dataset.from_pandas(df_partial).save_to_disk(ckpt_ds)
        print(f"Checkpoint saved at {i+1} images -> {ckpt_xlsx}")

Images to process: 200


 15%|█▌        | 30/200 [00:00<00:00, 296.84it/s]

Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]

 46%|████▌     | 91/200 [00:00<00:00, 211.03it/s]

Checkpoint saved at 50 images -> /content/batch_prompts_partial_50_images.xlsx


...

 75%|███████▌  | 150/200 [00:01<00:00, 82.04it/s]

Checkpoint saved at 150 images -> /content/batch_prompts_partial_150_images.xlsx


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

100%|██████████| 200/200 [00:01<00:00, 103.97it/s]

Checkpoint saved at 200 images -> /content/batch_prompts_partial_200_images.xlsx


In [ ]:
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
import io
from PIL import Image
import base64

In [ ]:
wb = Workbook()
ws = wb.active
ws.title = "batch_prompts"

# header
cols = ["index","file_name","prompt_type","generated_prompt","ground_truth","thumbnail"]
for c, h in enumerate(cols, 1):
    ws.cell(row=1, column=c, value=h)

rnum = 2
for _, r in df_batch.iterrows():
    ws.cell(row=rnum, column=1, value=int(r["index"]))
    ws.cell(row=rnum, column=2, value=r["file_name"])
    ws.cell(row=rnum, column=3, value=r["prompt_type"])
    ws.cell(row=rnum, column=4, value=r["generated_prompt"])
    ws.cell(row=rnum, column=5, value=r["ground_truth"])

    tb64 = r["thumbnail_b64"]
    if isinstance(tb64, str):
        try:
            img_bytes = base64.b64decode(tb64)
            img_buf = io.BytesIO(img_bytes)
            img_xl = XLImage(img_buf)
            ws.add_image(img_xl, f"F{rnum}")
        except Exception:
            pass

    rnum += 1

embedded_path = "/content/batch_prompts_1000_with_images.xlsx"
wb.save(embedded_path)
print("Saved embedded-image Excel:", embedded_path)

Saved embedded-image Excel: /content/batch_prompts_1000_with_images.xlsx


In [ ]:
# CELL 6
print("Prompt type counts:\n", df_batch["prompt_type"].value_counts())
print("Missing thumbs (rows where thumbnail_b64 is null):", df_batch["thumbnail_b64"].isna().sum())
# show 1 example per prompt type
for p in ["implicit","explicit","cot_template","autocot_caption","autocot_image"]:
    ex = df_batch[df_batch["prompt_type"]==p].iloc[0]
    print("\n=== example", p, "===")
    print("index:", ex["index"], "file:", ex["file_name"])
    print("prompt (truncated):", ex["generated_prompt"][:300].replace("\n"," "))

Prompt type counts:
 prompt_type
implicit           200
explicit           200
cot_template       200
autocot_caption    200
autocot_image      200
Name: count, dtype: int64
Missing thumbs (rows where thumbnail_b64 is null): 0

=== example implicit ===
index: 0 file: 000000561009.jpg
prompt (truncated): Describe the image.

=== example explicit ===
index: 0 file: 000000561009.jpg
prompt (truncated): Describe all objects, colors, actions, and spatial relationships in the image in clear and complete sentences.

=== example cot_template ===
index: 0 file: 000000561009.jpg
prompt (truncated): Think step-by-step about the image: 1. Identify all visible objects. 2. Describe their attributes (color, size, shape). 3. Explain spatial relationships between objects. 4. Identify any actions or interactions. 5. Infer the overall scene context. 6. Provide a final concise summary.

=== example autocot_caption ===
index: 0 file: 000000561009.jpg
prompt (truncated): Here's the chain-of-thought reasonin

In [34]:
!zip -r /content/batch_prompts_1000_dataset /content/sample_data

  adding: content/sample_data/ (stored 0%)
  adding: content/sample_data/anscombe.json (deflated 83%)
  adding: content/sample_data/README.md (deflated 39%)
  adding: content/sample_data/california_housing_train.csv (deflated 79%)
  adding: content/sample_data/california_housing_test.csv (deflated 76%)
  adding: content/sample_data/mnist_train_small.csv (deflated 88%)
  adding: content/sample_data/mnist_test.csv (deflated 88%)


In [35]:
!ls -R /content/batch_prompts_1000_dataset

/content/batch_prompts_1000_dataset:
data-00000-of-00001.arrow  dataset_info.json  state.json


In [36]:
!zip -r /content/batch_prompts_1000_dataset.zip /content/batch_prompts_1000_dataset

updating: content/batch_prompts_1000_dataset/ (stored 0%)
  adding: content/batch_prompts_1000_dataset/data-00000-of-00001.arrow (deflated 84%)
  adding: content/batch_prompts_1000_dataset/dataset_info.json (deflated 70%)
  adding: content/batch_prompts_1000_dataset/state.json (deflated 38%)


In [38]:
from google.colab import files
files.download('/content/batch_prompts_1000_dataset.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
!unzip -l /content/batch_prompts_1000_dataset.zip

Archive:  /content/batch_prompts_1000_dataset.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2025-11-26 07:52   content/batch_prompts_1000_dataset/
        0  2025-11-20 14:30   content/sample_data/
     1697  2000-01-01 08:00   content/sample_data/anscombe.json
      962  2000-01-01 08:00   content/sample_data/README.md
  1706430  2025-11-20 14:30   content/sample_data/california_housing_train.csv
   301141  2025-11-20 14:30   content/sample_data/california_housing_test.csv
 36523880  2025-11-20 14:30   content/sample_data/mnist_train_small.csv
 18289443  2025-11-20 14:30   content/sample_data/mnist_test.csv
 11458536  2025-11-26 07:52   content/batch_prompts_1000_dataset/data-00000-of-00001.arrow
      551  2025-11-26 07:52   content/batch_prompts_1000_dataset/dataset_info.json
      247  2025-11-26 07:52   content/batch_prompts_1000_dataset/state.json
---------                     -------
 68282887                     11 files
